# Region XIII (Caraga, PH16) — thresholds, activation RPs, and storm coverage

Notebook record of the 2026-07-31..08-03 threshold analysis. Inputs are on
blob `pa-aa-phl-storms/processed/`:
- `all_storms_hindcast_results/ensemble_500increments_region13/region13_ensemble_500increment_raw.csv`
  (23 storm-years x 80 thresholds; probability = fraction of <=12 ECMWF
  members whose region-summed predicted damage exceeds the threshold, one
  closest-approach cycle per storm)
- `besttrack_hindcast_results/per_storm_output/*_damage_table.csv` (38
  best-track runs, national tables -> Caraga aggregation)
- `caraga_storm_census/caraga_storm_census.csv` (IBTrACS 2006-2025 within
  100 km of PH16)

**Read the census README caveats first: both storm sets are floors** — the
ensemble list omits/zeroes Caraga's worst events (Rai scores 0.000 because
its selected cycle post-dates the Siargao landfall) and the best-track set
holds only 6 of 44 Caraga-relevant storms (no Rai/Washi/Tembin).

In [1]:
import io
import numpy as np
import ocha_stratus as stratus
import pandas as pd
BLOB = "pa-aa-phl-storms/processed"
r13 = pd.read_csv(io.BytesIO(stratus.load_blob_data(
    f"{BLOB}/all_storms_hindcast_results/ensemble_500increments_region13/region13_ensemble_500increment_raw.csv", stage="dev")))
census = pd.read_csv(io.BytesIO(stratus.load_blob_data(
    f"{BLOB}/caraga_storm_census/caraga_storm_census.csv", stage="dev")))
thr = [c for c in r13.columns if c.isdigit()]
YEARS = sorted(r13.year.unique()); span = max(YEARS) - min(YEARS) + 1
print(f"ensemble table: {len(r13)} storm-years, {len(thr)} thresholds, span {min(YEARS)}-{max(YEARS)} ({span} yrs)")
print(f"census: {len(census)} storms within 100 km; in ensemble run: {census.in_ensemble_region13.sum()}, in best-track set: {census.in_besttrack38.sum()}")

ensemble table: 23 storm-years, 80 thresholds, span 2006-2023 (18 yrs)
census: 44 storms within 100 km; in ensemble run: 11, in best-track set: 6


## Ensemble-based activation RPs (per probability cutoff x damage threshold)

In [2]:
rows = []
for k in range(1, 7):
    pcut = k/12 - 1e-9
    for t in thr:
        yrs = r13.loc[r13[t] >= pcut, "year"].nunique()
        rows.append({"members": k, "threshold": int(t), "act_years": yrs,
                     "RP": round(span/yrs, 1) if yrs else np.inf})
grid = pd.DataFrame(rows)
show = [500, 1000, 1500, 2000, 3000, 5000, 10000]
grid[grid.threshold.isin(show)].pivot(index="members", columns="threshold", values="RP")

threshold,500,1000,1500,2000,3000,5000,10000
members,,,,,,,
1,4.5,9.0,9.0,9.0,9.0,9.0,18.0
2,4.5,9.0,9.0,9.0,9.0,9.0,18.0
3,6.0,9.0,9.0,9.0,9.0,9.0,inf
4,18.0,18.0,18.0,inf,inf,inf,inf
5,18.0,18.0,18.0,inf,inf,inf,inf
6,18.0,18.0,inf,inf,inf,inf,inf


**Reading:** a 1-in-3-year activation exists ONLY as ">=500 houses, any
single member" (6 activation years in 18 -> RP 3.0). At >=50% member
probability nothing beyond 1,000 houses ever activates (RP 18). The
2025-regions-style table (95% of 3k ... 50% of 30k) NEVER activates for
Region XIII — max probability anywhere is 0.583 (Jelawat 2023) while Rai
reads zero (cycle-selection artifact).

## Best-track hazard basis (2006-2020, 38 national damage tables)

In [3]:
bt = []
blobs = [x for x in stratus.list_container_blobs(
    name_starts_with=f"{BLOB}/besttrack_hindcast_results/per_storm_output/", stage="dev")
    if x.endswith("damage_table.csv")]
for x in blobs:
    df = pd.read_csv(io.BytesIO(stratus.load_blob_data(x, stage="dev")))
    r = df[df.adm1_en.astype(str).str.contains("XIII", na=False)]
    n = x.split("/")[-1]
    bt.append({"storm": n.split("_")[0], "year": int(n.split("_")[1][:4]), "damage_median": r["median"].sum()})
bt = pd.DataFrame(bt)
ann = bt.groupby("year")["damage_median"].max()
years = pd.Series(0.0, index=range(bt.year.min(), bt.year.max()+1)); years.update(ann)
s = years.sort_values(ascending=False); n = len(s)
emp = pd.DataFrame({"year": s.index, "damage": s.values, "rank": np.arange(1, n+1)})
emp["RP_years"] = (n + 1) / emp["rank"]
nz = emp[emp.damage > 0]
print(nz.to_string(index=False))
for target in [3, 5, 10]:
    v = np.exp(np.interp(np.log(target), np.log(nz.RP_years.values[::-1]), np.log(nz.damage.values[::-1])))
    print(f"RP {target:>2}y -> ~{v:,.0f} damaged houses")

 year  damage  rank  RP_years
 2012 16608.0     1 16.000000
 2013  1603.0     2  8.000000
 2014   348.0     3  5.333333
 2016   174.0     4  4.000000
 2008    53.0     5  3.200000
 2019    48.0     6  2.666667
 2015     6.0     7  2.285714
RP  3y -> ~51 damaged houses
RP  5y -> ~298 damaged houses
RP 10y -> ~3,403 damaged houses


## Conclusions (as delivered 2026-08-03)

1. **Hazard basis**: RP3 ~ 50 damaged houses, RP5 ~ 300, RP10 ~ 3,400 —
   Bopha 2012 (16,608) is ~90% of all recorded Caraga damage. **These are
   floors**: the record misses Rai 2021 (best-track set ends 2020) and only
   6 of 44 Caraga-relevant storms are in it.
2. **A 1-in-3-yr forecast trigger for PH16 exists only at >=500 houses /
   any member** — high-confidence + high-damage combinations do not occur
   in this record.
3. Structural defects to fix before revisiting: per-region forecast-cycle
   selection (Rai), a Caraga-curated storm list (census on blob), and
   the pre-2021 TIGGE sparsity that makes Bopha/Washi unrecoverable
   locally. User decision 2026-08-03: existing thresholds accepted with
   these caveats recorded.